# Objetivo

Desenvolvimento do Teleutils em ambiente de teste local

# Configuração do Ambiente

## Bibliotecas

In [ ]:
# 1. Módulos da Biblioteca Padrão
import logging
from pathlib import Path

# 2. Módulos de Terceiros
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf
from pyspark.sql import types as T # ArrayType, StructType, StructField, StringType, BooleanType

# 3. Teletools/Teleutils
from teleutils.core.extractors import CDRParquetExtractor, CDRTextExtractor
from teleutils.core.transformers import CDRTransformer

In [ ]:
# Configuração do Pandas
# Exibir todas as colunas
pd.set_option("display.max_columns", None)

# Exibir todas as linhas
pd.set_option("display.max_rows", None)

# Evitar que o texto nas células seja cortado
pd.set_option("display.max_colwidth", None)

In [ ]:
# Configuração para exibir logs no output da célula
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)

# Reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("pyspark").setLevel(logging.WARNING)

## Spark

In [ ]:
spark = (
    SparkSession
        .builder
        .appName("SessaoLocal")
        .master("local[12]")
        .config("spark.driver.memory", "24g")
        .getOrCreate()
)
spark

## Arquivos de dados

### Dados de origem

In [ ]:
folder_algar = Path("/assets/monitoring/cdr/cdr_bruto/Semana101/Algar/FIXA/NGN")
_files_algar = [str(file) for file in folder_algar.glob("*.gz")]
df_algar = pd.DataFrame(_files_algar, columns=["file_path"])
df_algar["exchange_id"] = df_algar["file_path"].str.extract(r"/([A-Z]+)#")
files_algar = df_algar.groupby("exchange_id").sample(8,random_state=42)["file_path"].to_list()

In [ ]:
folder_claro = Path("/assets/monitoring/cdr/cdr_bruto/Semana101/Claro/home/mnt_srt2a/SRTEBT/coleta/coletados/")
_files_claro = [(str(file), str(file.parent.name)) for file in folder_claro.rglob("*HUAWEI/*.gz") if file.is_file()]
df_claro = pd.DataFrame(_files_claro, columns=["file_path", "exchange_id"])
files_claro = df_claro.groupby("exchange_id").sample(4,random_state=42)["file_path"].to_list()

### Arquivos intermediário e final

In [ ]:
extracted_algar = "/assets/data/cdr/teletools/db/extraidos/huawei_ngn_algar_extraido.parquet"
transformed_algar = "/assets/data/cdr/teletools/db/transformados/huawei_ngn_algar_transformado.parquet"

extracted_claro = "/assets/data/cdr/teletools/db/extraidos/huawei_ngn_claro_extraido.parquet"
transformed_claro = "/assets/data/cdr/teletools/db/transformados/huawei_ngn_claro_transformado.parquet"

# Análise

## Algar

### Dados brutos

In [ ]:
df = spark.read.csv(files_algar, sep=",")
df.limit(5).toPandas()

### Extração

In [ ]:
extractor = CDRTextExtractor(spark)
extractor.extract(files_algar, extracted_algar, "stfc_huawei_ngn")

In [ ]:
df_extracted = spark.read.parquet(extracted_algar)
df_extracted.show(5)

### Transformação

In [ ]:
transformer = CDRTransformer(spark)
transformer.transform_stfc_huawei_ngn(extracted_algar, transformed_algar)

In [ ]:
df_transformed = spark.read.parquet(transformed_algar)
df_transformed.limit(5).toPandas().dropna(how="all",axis=1)

## Claro

### Dados brutos

In [ ]:
df = spark.read.csv(files_claro, sep=",")
df.limit(5).toPandas()

### Extração

In [ ]:
extractor = CDRTextExtractor(spark)
extractor.extract(files_algar, extracted_algar, "stfc_huawei_ngn")

In [ ]:
df_extracted = spark.read.parquet(extracted_algar)
df_extracted.show(5)

### Transformação

In [ ]:
transformer = CDRTransformer(spark)
transformer.transform_stfc_huawei_ngn(extracted_algar, transformed_algar)

In [ ]:
df_transformed = spark.read.parquet(transformed_algar)
df_transformed.limit(5).toPandas().dropna(how="all",axis=1)